In [14]:
import os
os.chdir(r"C:\Users\chang\OneDrive\Documents\code\probingsycophancy")

import yaml
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sycophancy.activations import ActivationExtractor

In [15]:

def load_config(path):
    with open(path) as f:
        return yaml.safe_load(f)


def load_model(config):
    model = AutoModelForCausalLM.from_pretrained(
        config["model_id"],
        torch_dtype=getattr(torch, config.get("dtype", "float32")),
    )
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(config["model_id"])
    return model, tokenizer


In [16]:
# Load config
with open("configs/models/tinyllama_1b.yaml") as f:
    config = yaml.safe_load(f)

In [17]:
config["components"]

['attn', 'mlp', 'residual']

In [18]:
# Load model
model = AutoModelForCausalLM.from_pretrained(config["model_id"], torch_dtype=torch.float32)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(config["model_id"])


Loading weights: 100%|██████████| 201/201 [00:03<00:00, 51.02it/s]


In [ ]:
# Test hooks
extractor = ActivationExtractor(model, tokenizer, config)
extractor.attach_hooks(range(10, 22), components= config["components"])

In [ ]:
# Run a prompt
prompts = ["The capital of France is"]
activations = extractor.extract(prompts) # at last token

for key, tensor in activations.items():
    print(f"{key}: {tensor.shape}")

(10, 'attn'): torch.Size([1, 32, 64])
(11, 'attn'): torch.Size([1, 32, 64])
(12, 'attn'): torch.Size([1, 32, 64])
(13, 'attn'): torch.Size([1, 32, 64])
(14, 'attn'): torch.Size([1, 32, 64])
(15, 'attn'): torch.Size([1, 32, 64])
(16, 'attn'): torch.Size([1, 32, 64])
(17, 'attn'): torch.Size([1, 32, 64])
(18, 'attn'): torch.Size([1, 32, 64])
(19, 'attn'): torch.Size([1, 32, 64])
(20, 'attn'): torch.Size([1, 32, 64])
(21, 'attn'): torch.Size([1, 32, 64])


In [ ]:
extractor.cleanup()